# 08 — Fine-tuning de Transformers em GPU (Google Colab / Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/luanfreitas5/sentimento-ptbr-llm/blob/main/notebooks/08_cloud_gpu_transformer.ipynb)

Executa a etapa **`training_transformer`** (BERTimbau, RoBERTa/BR_BERTo e DistilBERT) em uma GPU
gratuita, reaproveitando exatamente o código do repositório
(`pipelines.training_transformer.run_training_transformer_stage` e `configs/model_params.yaml`).
O mesmo notebook funciona no **Colab** e no **Kaggle** (a plataforma é detectada sozinha).

## Antes de começar

| | Colab | Kaggle |
|---|---|---|
| GPU | Runtime → Change runtime type → **T4 GPU** | Settings → Accelerator → **GPU T4 x2** ou **P100** |
| Dados | Suba `treino.parquet`, `validacao.parquet` e `teste.parquet` (de `data/gold/`) para o Drive em `MyDrive/sentimento-ptbr-llm/data/gold/` | Crie um **Dataset** com esses 3 arquivos e anexe-o (Add Input) |
| Internet | já habilitada | Settings → **Internet On** (necessária para clonar o repo e baixar os modelos do Hugging Face) |
| Importar | Abra pelo botão acima | File → Import Notebook → GitHub |
| Resultados | Salvos no Drive (`MyDrive/sentimento-ptbr-llm/outputs/`) | Aba **Output** (`/kaggle/working/outputs/`); use *Save Version → Save & Run All* para rodar sem a aba aberta |

**Primeiro rode com `SMOKE_TEST = True`** (poucos dados, 1 época) para validar o pipeline e só então
troque para `False`. As sessões gratuitas desconectam: cada modelo é salvo assim que termina e, ao
reexecutar, os já treinados são reaproveitados (`FORCE_RETRAIN = False`).

> **Aviso:** o treino roda em precisão float32 (o código de `src/models/base.py` não usa `fp16`).
> Ajuste `EPOCHS`/`BATCH_SIZE` conforme a cota de GPU; meça o tempo da época no smoke test antes do treino completo.

## 1. Plataforma e GPU

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"Plataforma detectada: {PLATFORM}")

import torch

if torch.cuda.is_available():
    gpu_properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_properties.name} | VRAM: {gpu_properties.total_memory / 1e9:.1f} GB")
else:
    print(
        "ATENÇÃO: nenhuma GPU detectada. Colab: Runtime > Change runtime type > GPU. "
        "Kaggle: Settings > Accelerator > GPU T4 x2 / P100."
    )

## 2. Parâmetros da execução

In [ ]:
# --- Repositório -----------------------------------------------------------
REPO_URL = "https://github.com/luanfreitas5/sentimento-ptbr-llm.git"
REPO_BRANCH = "main"

# --- Onde estão os dados (treino/validacao/teste.parquet) -------------------
# Colab : pasta do Google Drive que contém treino.parquet, validacao.parquet e teste.parquet
#         (em qualquer subpasta). Caminho relativo a /content/drive/MyDrive.
# Kaggle: os arquivos devem estar num Dataset anexado ao notebook (Add Input);
#         o notebook procura em /kaggle/input/* automaticamente.
DRIVE_DATA_DIR = "sentimento-ptbr-llm/data/gold"
DRIVE_OUTPUT_DIR = "sentimento-ptbr-llm/outputs"  # Colab: onde salvar os resultados

# --- Execução ---------------------------------------------------------------
NOTEBOOK_TAG = "transformer"
MODEL_NAMES = [
    "bertimbau",
    "roberta",
    "distilbert",
]  # chaves de configs/model_params.yaml -> transformers

SMOKE_TEST = (
    True  # True: subconjunto pequeno + 1 época (valida o pipeline). False: corpus completo.
)
SMOKE_TRAIN_SIZE = 600
SMOKE_EVAL_SIZE = 200

EPOCHS = None  # None = usa o YAML (4). Ex.: 2 para economizar cota de GPU.
BATCH_SIZE = None  # None = usa o YAML (16/32). Reduza (8) se der "CUDA out of memory".
MAX_LENGTH = None  # None = usa o YAML (128).
FORCE_RETRAIN = False  # True: ignora checkpoints já salvos em PERSIST_DIR.
TRACK_WITH_MLFLOW = False  # True: registra no MLflow local (sqlite) — opcional na nuvem.
AUTO_DISCONNECT = (
    False  # Colab: True encerra o runtime ao terminar (economiza unidades de computação).
)

## 3. Repositório, dados e pastas
Clona o projeto, copia os parquets para `data/gold/` e prepara a pasta de resultados.

In [ ]:
# --- Pastas de trabalho ----------------------------------------------------
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = WORK_DIR / "sentimento-ptbr-llm"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_SEARCH_DIRS = [Path("/content/drive/MyDrive") / DRIVE_DATA_DIR, Path("/content/data")]
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR
elif IN_KAGGLE:
    DATA_SEARCH_DIRS = [path for path in Path("/kaggle/input").glob("*") if path.is_dir()]
    OUTPUT_ROOT = Path("/kaggle/working/outputs")  # tudo em /kaggle/working vira output do notebook
else:
    DATA_SEARCH_DIRS = [REPO_DIR / "data" / "gold"]
    OUTPUT_ROOT = WORK_DIR / "outputs"

# Separa execução de teste rápido (smoke) da execução completa: evita reaproveitar
# por engano um checkpoint treinado com poucos dados.
RUN_TAG = "smoke" if SMOKE_TEST else "full"
PERSIST_DIR = OUTPUT_ROOT / NOTEBOOK_TAG / RUN_TAG
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Resultados serão salvos em: {PERSIST_DIR}")


def get_secret(name: str) -> str | None:
    """Lê um segredo (Colab: Secrets; Kaggle: Add-ons > Secrets; senão, variável de ambiente)."""
    try:
        if IN_COLAB:
            from google.colab import userdata

            return userdata.get(name)
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret(name)
    except Exception:  # segredo inexistente ou sem acesso concedido ao notebook
        return None
    return os.environ.get(name)


# --- Clona (ou atualiza) o repositório --------------------------------------
if not (REPO_DIR / ".git").exists():
    token = get_secret("GITHUB_TOKEN")  # só necessário se o repositório for privado
    clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)],
        check=True,
        capture_output=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
print(
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
        check=False,
        capture_output=True,
        text=True,
    ).stdout
)

# --- Copia os parquets de dados para data/gold do repositório ---------------
GOLD_DIR = REPO_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
for file_name in ("treino.parquet", "validacao.parquet", "teste.parquet"):
    found = next(
        (match for base in DATA_SEARCH_DIRS if base.exists() for match in base.rglob(file_name)),
        None,
    )
    if found is None:
        raise FileNotFoundError(
            f"'{file_name}' não encontrado em {[str(p) for p in DATA_SEARCH_DIRS]}. "
            "Envie os parquets de data/gold/ para o Drive (Colab) ou crie um Dataset (Kaggle)."
        )
    if found.resolve() != (GOLD_DIR / file_name).resolve():
        shutil.copy2(found, GOLD_DIR / file_name)
    print(f"OK  {file_name}  <-  {found}")

sys.path.insert(0, str(REPO_DIR / "src"))  # o projeto importa módulos a partir de src/
os.chdir(REPO_DIR)

## 4. Dependências
Torch e Transformers já vêm instalados no Colab/Kaggle; instalamos só o restante usado pelo projeto.
Se o pip pedir para reiniciar a sessão, reinicie e execute novamente a partir da célula 1.

In [ ]:
import subprocess
import sys

PACKAGES = [
    "pydantic",
    "pydantic-settings",
    "python-dotenv",
    "pyyaml",
    "joblib",
    "pandas",
    "pandera[polars]",
    "polars",
    "pyarrow",
    "scikit-learn",
    "scipy",
    "statsmodels",
    "xgboost",
    "mlflow",
    "rich",
    "matplotlib",
    "seaborn",
    "transformers>=4.44",
    "accelerate>=1.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

## 5. Carga dos dados

In [ ]:
import logging
from collections import Counter

from rich.logging import RichHandler

from config.constants import CONFIG_FILE_NAMES
from config.paths import CONFIGS_DIR, load_project_paths
from data.loader import load_training_example_dataset
from io_utils.yaml import read_yaml

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[RichHandler(rich_tracebacks=True)],
    force=True,
)

paths = load_project_paths()
paths.models_checkpoints_dir.mkdir(parents=True, exist_ok=True)


def load_split(path: Path) -> tuple[list[str], list[str]]:
    """Lê textos e rótulos de uma partição parquet."""
    corpus = load_training_example_dataset(path)
    return corpus["text"].to_list(), corpus["sentiment_label"].to_list()


X_train, y_train = load_split(paths.training_corpus_file)
X_val, y_val = load_split(paths.validation_corpus_file)
X_test, y_test = load_split(paths.test_corpus_file)

for split_name, labels in (("treino", y_train), ("validação", y_val), ("teste", y_test)):
    print(f"{split_name:>10}: {len(labels):>7} textos | {dict(Counter(labels))}")

if SMOKE_TEST:
    from sklearn.model_selection import train_test_split

    def stratified_subset(texts, labels, size):
        """Subconjunto estratificado e determinístico."""
        if size >= len(texts):
            return texts, labels
        sub_x, _, sub_y, _ = train_test_split(
            texts, labels, train_size=size, stratify=labels, random_state=42
        )
        return sub_x, sub_y

    X_train, y_train = stratified_subset(X_train, y_train, SMOKE_TRAIN_SIZE)
    X_val, y_val = stratified_subset(X_val, y_val, SMOKE_EVAL_SIZE)
    X_test, y_test = stratified_subset(X_test, y_test, SMOKE_EVAL_SIZE)
    print(f"SMOKE_TEST ativo: treino={len(X_train)}, validação={len(X_val)}, teste={len(X_test)}")

## 6. Hiperparâmetros
Lidos de `configs/model_params.yaml` e, opcionalmente, sobrescritos acima.

In [ ]:
transformer_params = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["model_params"])["transformers"]
model_params = {name: dict(transformer_params[name]) for name in MODEL_NAMES}

for params in model_params.values():
    if SMOKE_TEST:
        params["epochs"] = 1
    elif EPOCHS is not None:
        params["epochs"] = EPOCHS
    if BATCH_SIZE is not None:
        params["batch_size"] = BATCH_SIZE
    if MAX_LENGTH is not None:
        params["max_length"] = MAX_LENGTH

for name, params in model_params.items():
    print(
        name,
        {
            key: params[key]
            for key in ("model_name", "epochs", "batch_size", "max_length", "learning_rate")
        },
    )

## 7. Treino, avaliação no teste e persistência (um modelo por vez)
Cada modelo é treinado, avaliado, salvo em `PERSIST_DIR` e liberado da GPU antes do próximo. Se a sessão cair, basta reexecutar: os modelos já salvos são reaproveitados.

In [ ]:
import gc
import time

import polars as pl

from evaluation.evaluator import evaluate_classifier
from evaluation.reports import (
    build_evaluation_report,
    merge_evaluation_reports,
    save_evaluation_report,
)
from models.persistence import load_classifier
from pipelines.training_transformer import run_training_transformer_stage
from visualization.confusion_matrix import plot_confusion_matrix_heatmap
from visualization.theme import apply_project_theme, save_figure

apply_project_theme()
persist_checkpoints = PERSIST_DIR / "checkpoints"
persist_figures = PERSIST_DIR / "figures"
persist_reports = PERSIST_DIR / "reports"
for folder in (persist_checkpoints, persist_figures, persist_reports):
    folder.mkdir(parents=True, exist_ok=True)

evaluation_reports, evaluation_results, training_seconds = [], {}, {}

for model_name in MODEL_NAMES:
    saved_checkpoint = persist_checkpoints / f"{model_name}.pt"
    local_checkpoint = paths.models_checkpoints_dir / f"{model_name}.pt"

    if saved_checkpoint.exists() and not FORCE_RETRAIN:
        print(f"[{model_name}] checkpoint encontrado em {saved_checkpoint}; pulando o treino.")
        shutil.copy2(saved_checkpoint, local_checkpoint)
        model = load_classifier(local_checkpoint, backend="torch")
        training_seconds[model_name] = float("nan")
    else:
        print(f"[{model_name}] iniciando fine-tuning...")
        started_at = time.perf_counter()
        results = run_training_transformer_stage(
            X_train,
            y_train,
            X_val,
            y_val,
            model_names=(model_name,),
            model_params={model_name: model_params[model_name]},
            checkpoints_dir=paths.models_checkpoints_dir,
            track_with_mlflow=TRACK_WITH_MLFLOW,
        )
        training_seconds[model_name] = time.perf_counter() - started_at
        model = results[model_name].model
        minutes = training_seconds[model_name] / 60
        metrics = results[model_name].metrics
        print(f"[{model_name}] treino concluído em {minutes:.1f} min | validação: {metrics}")
        shutil.copy2(
            local_checkpoint, saved_checkpoint
        )  # persiste já: a sessão pode cair a qualquer momento

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)
    result = evaluate_classifier(y_test, y_pred, y_score=y_score)
    evaluation_results[model_name] = result
    print(f"[{model_name}] TESTE: {result.point_metrics}")
    print(f"[{model_name}] IC95% F1-macro: {result.confidence_intervals.get('f1_macro')}")

    figure = plot_confusion_matrix_heatmap(
        result.confusion_matrix, title=f"Matriz de Confusão — {model_name}"
    )
    save_figure(figure, f"matriz_confusao_{model_name}", directory=persist_figures)
    evaluation_reports.append(build_evaluation_report(model_name, result))

    pl.DataFrame({"text": X_test, "y_true": y_test, "y_pred": list(y_pred)}).write_parquet(
        persist_reports / f"predicoes_{model_name}.parquet"
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()

report = merge_evaluation_reports(evaluation_reports)
save_evaluation_report(report, persist_reports / "avaliacao_transformers.csv")
display(report)

## 8. Comparação e salvamento
Tempos de treino por modelo e empacotamento dos resultados para download.

In [ ]:
print(
    {
        name: ("reaproveitado" if math.isnan(seconds) else round(seconds / 60, 1))
        for name, seconds in training_seconds.items()
    }
)

import math

archive_base = WORK_DIR / f"resultados_{NOTEBOOK_TAG}_{RUN_TAG}"
archive_path = shutil.make_archive(str(archive_base), "zip", PERSIST_DIR)
print(f"Pacote com checkpoints, figuras e métricas: {archive_path}")
print("Colab : os arquivos já estão no Drive. Kaggle: use Save Version e baixe em Output.")

if IN_COLAB and not SMOKE_TEST:
    from google.colab import files

    # files.download(archive_path)  # descomente para baixar também para o computador

## 9. Próximos passos (no seu computador)

1. Baixe `checkpoints/*.pt` e copie para `models/checkpoints/` do repositório local.
2. Rode `make evaluate` — a etapa `evaluate` descobre os checkpoints em `models/checkpoints` e faz a
   comparação estatística completa (McNemar, bootstrap, calibração, fatias) contra os demais modelos.
3. Para rodar a etapa **LLM**, use `09_cloud_gpu_llm.ipynb`.

> Esta avaliação (seção 7) é exploratória; os números oficiais do trabalho vêm do estágio `evaluate`.
> Quando `SMOKE_TEST = True` os resultados **não** devem ser reportados.

In [ ]:
if AUTO_DISCONNECT and IN_COLAB:
    from google.colab import runtime

    runtime.unassign()  # libera a GPU